# Chapter 4: Building the GPT Architecture

In Chapter 3 we built multi-head causal self-attention — the mechanism that lets 
tokens relate to each other. Now we build the full GPT model around it.

This chapter adds four components:

1. **Layer Normalization** — stabilizes values between layers
2. **GELU Activation** — smooth non-linearity
3. **Feed-Forward Network** — per-token processing
4. **Residual Connections** — skip connections to prevent vanishing gradients

These combine into a **Transformer Block**, and stacking multiple blocks gives us GPT.

## 4.1 Layer Normalization — The Problem

As values pass through many layers, they can grow or shrink uncontrollably. 
LayerNorm fixes this by normalizing each token vector to mean ≈ 0 and variance ≈ 1.

First, let's see what a token vector looks like before normalization.

In [21]:
import torch
import torch.nn as nn

torch.manual_seed(123)

# Fake input: batch of 2 sequences, 6 tokens each, embedding dim 4
# Shape [B, T, emb_dim] — same shape as output from Ch2 embeddings
x = torch.rand(2, 6, 4)

# Inspect one token vector — mean and variance are arbitrary
print("Before:", x[0, 0])
print("Mean:", x[0, 0].mean().item())   # not 0
print("Var:", x[0, 0].var().item())     # not 1

Before: tensor([0.2961, 0.5166, 0.2517, 0.6886])
Mean: 0.4382254183292389
Var: 0.041267119348049164


## 4.2 Layer Normalization — PyTorch Built-in

`nn.LayerNorm(4)` normalizes each token vector of size 4 to mean ≈ 0 and variance ≈ 1.

After normalization, values are in a stable range regardless of what the previous 
layer produced. This makes training much more reliable in deep networks.

In [22]:
# PyTorch's built-in LayerNorm — normalizes vectors of size emb_dim=4
ln = nn.LayerNorm(4)

out = ln(x)

# Mean should be ≈ 0, variance should be ≈ 1
print("After:", out[0, 0])
print("Mean:", out[0, 0].mean().item())
print("Var:", out[0, 0].var(correction=0).item())

After: tensor([-0.8077,  0.4452, -1.0602,  1.4227], grad_fn=<SelectBackward0>)
Mean: -5.960464477539063e-08
Var: 0.9996770620346069


## 4.3 Layer Normalization — From Scratch

Building LayerNorm manually to understand the internals. Four steps:

1. Compute mean of each token vector
2. Compute variance of each token vector
3. Normalize: `(x - mean) / sqrt(var + eps)`
4. Apply learnable `scale` and `shift`

`scale` and `shift` are `nn.Parameter` — they let the model adjust the 
normalization if strict mean=0, var=1 is too restrictive. They start at 
`ones` and `zeros` (no effect), but training can adjust them via backprop.

`eps = 1e-5` prevents division by zero if variance happens to be 0.

`keepdim=True` preserves the reduced dimension for broadcasting — mean is 
`[B, T, 1]` not `[B, T]`, so subtraction from `[B, T, emb_dim]` works.

In [23]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5  # prevents division by zero

        # Learnable parameters — let the model adjust normalization if needed
        # scale starts at 1 (no scaling), shift starts at 0 (no shifting)
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        # Compute per-token stats along embedding dim
        # keepdim=True keeps shape [B, T, 1] for broadcasting
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, correction=0)

        # Normalize to mean=0, var=1
        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        # Apply learnable scale and shift
        return self.scale * x_norm + self.shift

# Verify it matches PyTorch's built-in
ln2 = LayerNorm(4)
out2 = ln2(x)
print("Mean:", out2[0, 0].mean().item())   # ≈ 0
print("Var:", out2[0, 0].var(correction=0).item())   # ≈ 1

Mean: 1.1920928955078125e-07
Var: 0.9996770024299622


## 4.4 GELU Activation

Activation functions add non-linearity — without them, stacking linear layers 
just produces another linear layer.

ReLU is the classic choice: negative values become 0, positives pass through. 
Simple but harsh — information from negative values is completely destroyed.

GELU (Gaussian Error Linear Unit) is smoother. Small negatives get dampened 
instead of killed, giving the model more nuance. GPT-2 uses GELU throughout.

In [24]:
class GELU(nn.Module):
    def forward(self, x):
        # Approximate GELU formula used in GPT-2
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * x**3)
        ))

# Compare GELU vs ReLU on negative and positive values
gelu = GELU()

print("ReLU(-1):", torch.relu(torch.tensor(-1.0)).item())   # killed to 0
print("GELU(-1):", gelu(torch.tensor(-1.0)).item())          # dampened, not killed
print("ReLU(2):", torch.relu(torch.tensor(2.0)).item())      # unchanged
print("GELU(2):", gelu(torch.tensor(2.0)).item())            # nearly unchanged

ReLU(-1): 0.0
GELU(-1): -0.15880799293518066
ReLU(2): 2.0
GELU(2): 1.9545977115631104


## 4.5 Feed-Forward Network

Attention relates tokens to each other. The feed-forward network processes each 
token independently — it gives the model per-token "thinking" capacity.

Two linear layers with GELU in between:

In [25]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        # Expand to 4x, apply non-linearity, compress back
        # nn.Sequential chains layers — runs them in order
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),   # expand
            GELU(),                              # non-linearity
            nn.Linear(4 * emb_dim, emb_dim)     # compress back
        )

    def forward(self, x):
        return self.layers(x)  # each token processed independently

ff = FeedForward(emb_dim=4)
out = ff(x)
print("Input shape:", x.shape)    # [2, 6, 4]
print("Output shape:", out.shape)  # [2, 6, 4] — same shape in and out

Input shape: torch.Size([2, 6, 4])
Output shape: torch.Size([2, 6, 4])


## 4.6 All Components — Dependencies

This cell collects all building blocks needed for the Transformer Block and 
full GPT model. Each was built incrementally in earlier sections:

- **GELU** — smooth activation (Section 4.4)
- **LayerNorm** — normalizes token vectors to stable range (Section 4.3)
- **FeedForward** — per-token processing with 4x expansion (Section 4.5)
- **MultiHeadAttention** — from Chapter 3, relates tokens to each other

These are imported here so the notebook runs standalone.

In [6]:
import torch
import torch.nn as nn

torch.manual_seed(123)

# --- GELU Activation (Section 4.4) ---
class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * x**3)
        ))

# --- Layer Normalization (Section 4.3) ---
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, correction=0)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x_norm + self.shift

# --- Feed-Forward Network (Section 4.5) ---
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            GELU(),
            nn.Linear(4 * emb_dim, emb_dim)
        )

    def forward(self, x):
        return self.layers(x)

# --- Multi-Head Attention (Chapter 3) ---
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads):
        super().__init__()
        assert d_out % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_q = nn.Linear(d_in, d_out, bias=False)
        self.W_k = nn.Linear(d_in, d_out, bias=False)
        self.W_v = nn.Linear(d_in, d_out, bias=False)
        self.W_o = nn.Linear(d_out, d_out, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        B, T, d_in = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = Q @ K.transpose(2, 3) / self.head_dim**0.5
        attn_scores = attn_scores.masked_fill(self.mask.bool()[:T, :T], float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = attn_weights @ V
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(out)

## 4.7 Transformer Block

The core repeating unit of GPT. Combines everything built so far:

In [26]:
class TransformerBlock(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout):
        super().__init__()
        self.att = MultiHeadAttention(d_in, d_out, context_length, dropout, num_heads)
        self.ff = FeedForward(d_out)
        self.norm1 = LayerNorm(d_out)   # norm before attention
        self.norm2 = LayerNorm(d_out)   # norm before feed-forward
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # Attention block: normalize → attend → dropout → add residual
        x = x + self.drop(self.att(self.norm1(x)))

        # Feed-forward block: normalize → process → dropout → add residual
        x = x + self.drop(self.ff(self.norm2(x)))

        return x

# Test — output shape matches input (shape-preserving)
torch.manual_seed(123)
x = torch.rand(2, 6, 4)  # [batch, seq_len, emb_dim]
block = TransformerBlock(d_in=4, d_out=4, context_length=6, num_heads=2, dropout=0.0)
out = block(x)
print("Input shape:", x.shape)    # [2, 6, 4]
print("Output shape:", out.shape)  # [2, 6, 4]

Input shape: torch.Size([2, 6, 4])
Output shape: torch.Size([2, 6, 4])


## 4.8 Full GPT Model

The complete GPT architecture — everything from Chapters 2-4 combined:

1. **Token embeddings** — converts token IDs to vectors (Ch2)
2. **Positional embeddings** — adds position information (Ch2)
3. **Dropout** — regularization on input embeddings
4. **Transformer blocks** — stacked `num_layers` times (Ch3 + Ch4)
5. **Final LayerNorm** — stabilize before output projection
6. **Output head** — projects from `emb_dim` back to `vocab_size`

The output is `[B, T, vocab_size]` — at each position, a score over every 
token in the vocabulary. The highest score is the model's prediction for the 
next token. This is a 50,257-class classification problem at every position.

The config below matches GPT-2 (124M parameters): `emb_dim=768`, 
`num_heads=12`, `num_layers=12`, `context_length=1024`.

In [29]:
class GPTModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, context_length, num_heads, num_layers, dropout):
        super().__init__()

        # Ch2 — token and positional embeddings
        self.tok_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(context_length, emb_dim)
        self.drop = nn.Dropout(dropout)

        # Stack num_layers transformer blocks
        # * unpacks the list so nn.Sequential gets individual blocks
        self.blocks = nn.Sequential(
            *[TransformerBlock(emb_dim, emb_dim, context_length, num_heads, dropout)
              for _ in range(num_layers)]
        )

        # Final normalization before output projection
        self.norm = LayerNorm(emb_dim)

        # Project from emb_dim to vocab_size — one score per vocabulary token
        self.out_head = nn.Linear(emb_dim, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape  # batch size, sequence length

        tok_emb = self.tok_emb(idx)                                # [B, T, emb_dim]
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device)) # [T, emb_dim]
        x = self.drop(tok_emb + pos_emb)                           # [B, T, emb_dim]
        x = self.blocks(x)                                         # [B, T, emb_dim]
        x = self.norm(x)                                           # [B, T, emb_dim]
        logits = self.out_head(x)                                  # [B, T, vocab_size]
        return logits

# GPT-2 config: 124M parameters
torch.manual_seed(123)
model = GPTModel(
    vocab_size=50257,
    emb_dim=768,
    context_length=1024,
    num_heads=12,
    num_layers=12,
    dropout=0.1
)

# Test with fake token IDs
dummy_input

tensor([[46228,  2012, 45911, 29735, 15545, 47804, 31472, 45851, 32511,  9942,
          4604,  8952, 17930, 34880, 12630, 30920]])

## 4.9 Greedy Text Generation

The generation loop feeds output back as input, one token at a time. This is 
**autoregressive** generation — each predicted token becomes part of the input 
for predicting the next.

At each step:
1. Run the full model on the current sequence
2. Take only the last position's logits — that's the prediction for the next token
3. Pick the token with the highest score (`argmax`) — this is **greedy** decoding
4. Append it to the sequence
5. Repeat

The model is untrained, so output will be random token IDs. After training 
(Chapter 5), these become actual coherent words.

In [30]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        logits = model(idx)                                # [B, T, vocab_size]
        logits = logits[:, -1, :]                          # last position: [B, vocab_size]
        idx_next = logits.argmax(dim=-1, keepdim=True)     # highest score: [B, 1]
        idx = torch.cat([idx, idx_next], dim=1)            # append: [B, T+1]
    return idx

# Test with fake starting tokens — output is nonsense (untrained model)
torch.manual_seed(123)
start = torch.tensor([[1, 2, 3]])
output = generate(model, start, max_new_tokens=5)
print("Generated token IDs:", output)

Generated token IDs: tensor([[    1,     2,     3, 40283,  6464, 25816, 34065, 36903]])


# Chapter 5: Training the GPT Model

We now have the full GPT architecture from Chapter 4. This chapter makes it learn.

The pipeline:
1. **Data loading** — chunk text into input-target pairs
2. **Loss function** — cross-entropy measures how wrong predictions are
3. **Training loop** — forward pass, compute loss, backprop, update weights
4. **Text generation** — generate coherent text from the trained model

## 5.1 GPT Dataset

Training data for a language model is simple: the target is the input shifted 
by one token. At every position, the model learns to predict the next token.

In [31]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize entire text into token IDs
        token_ids = tokenizer.encode(txt)

        # Slide a window across the text to create input-target pairs
        for i in range(0, len(token_ids) - context_length, stride):
            input_chunk = token_ids[i:i + context_length]          # tokens [i, i+T)
            target_chunk = token_ids[i + 1:i + context_length + 1]  # tokens [i+1, i+T+1)
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

## 5.2 Data Loader

Wraps the dataset into a PyTorch `DataLoader` which handles batching and 
shuffling automatically.

We use `tiktoken` — OpenAI's tokenizer for GPT-2 — with a vocabulary of 
50,257 tokens. This matches our model's `vocab_size`.

The test text is a single sentence repeated 100 times. The model should 
be able to memorize this easily — a sanity check before training on real data.

In [32]:
import tiktoken

def create_dataloader(txt, batch_size=4, context_length=256, stride=128):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(txt, tokenizer, context_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader

# Test with repeated text — easy memorization target
text = "Every effort moves you forward. Every effort moves you forward. " * 100
dl = create_dataloader(text, batch_size=2, context_length=16, stride=16)

batch = next(iter(dl))
inputs, targets = batch
print("Inputs shape:", inputs.shape)    # [2, 16]
print("Targets shape:", targets.shape)  # [2, 16] — shifted by 1
print("Inputs[0]:", inputs[0])
print("Targets[0]:", targets[0])

Inputs shape: torch.Size([2, 16])
Targets shape: torch.Size([2, 16])
Inputs[0]: tensor([3887, 3626, 6100,  345, 2651,   13, 3887, 3626, 6100,  345, 2651,   13,
        3887, 3626, 6100,  345])
Targets[0]: tensor([3626, 6100,  345, 2651,   13, 3887, 3626, 6100,  345, 2651,   13, 3887,
        3626, 6100,  345, 2651])


## 5.3 Model Setup

Full GPT-2 (124M params) won't fit on a 4GB GPU for training. We use a 
smaller config — same architecture, reduced dimensions:

| Config      | GPT-2   | Ours    |
|-------------|---------|---------|
| emb_dim     | 768     | 128     |
| num_heads   | 12      | 4       |
| num_layers  | 12      | 4       |
| context_len | 1024    | 256     |
| params      | 124M    | ~13.7M  |

`.to(device)` moves all parameters to GPU for faster training.

In [33]:
torch.manual_seed(123)
device = torch.device("cuda")

# Smaller config that fits on 4GB VRAM
model = GPTModel(
    vocab_size=50257,
    emb_dim=128,
    context_length=256,
    num_heads=4,
    num_layers=4,
    dropout=0.1
).to(device)  # move all weights to GPU

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {next(model.parameters()).device}")

Parameters: 13,689,856
Device: cuda:0


## 5.4 Training Loop

The core training loop repeats three steps for every batch:

1. **Forward pass** — run input through model, get logits `[B, T, vocab_size]`
2. **Compute loss** — cross-entropy between predictions and targets
3. **Backward pass** — compute gradients and update weights

**AdamW** is the standard optimizer for transformers. It adapts the learning 
rate per parameter and includes weight decay for regularization.

**Cross-entropy** expects shapes `[N, classes]` and `[N]`, so we flatten:
- logits: `[B, T, vocab_size]` → `[B*T, vocab_size]`
- targets: `[B, T]` → `[B*T]`

Each of the `B*T` positions is an independent classification problem — 
"which of 50,257 tokens comes next?"

**`loss.backward()`** computes gradients for every `nn.Parameter` in the model.
**`optimizer.step()`** nudges all parameters in the direction that reduces loss.
**`optimizer.zero_grad()`** resets gradients so they don't accumulate across batches.

In [34]:
# AdamW — standard transformer optimizer
# lr=4e-4 is a common starting learning rate
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4)

model.train()  # enable dropout

for epoch in range(5):
    for batch_idx, (inputs, targets) in enumerate(dl):
        # Move data to GPU
        inputs, targets = inputs.to(device), targets.to(device)

        # Forward pass
        logits = model(inputs)  # [B, T, vocab_size]

        # Cross-entropy needs [B*T, vocab_size] and [B*T]
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, logits.shape[-1]),  # flatten to [B*T, vocab_size]
            targets.view(-1)                     # flatten to [B*T]
        )

        # Backward pass — compute gradients, update weights, reset
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 3.4328
Epoch 2, Loss: 0.5793
Epoch 3, Loss: 0.1292
Epoch 4, Loss: 0.0651
Epoch 5, Loss: 0.0417


## 5.5 Text Generation

After training, we switch to `model.eval()` — disables dropout so output is 
deterministic. `torch.no_grad()` skips gradient computation since we're not 
training — saves memory and speeds up inference.

The `generate()` function from Chapter 4 now produces actual words instead of 
random token IDs. Since we trained on "Every effort moves you forward" repeated, 
the model should reproduce that pattern.

This is a memorization test — not real generalization. Training on a larger, 
diverse corpus would produce more interesting output.

In [35]:
tokenizer = tiktoken.get_encoding("gpt2")

model.eval()  # disable dropout for deterministic output

start_text = "Every effort"
start_ids = torch.tensor([tokenizer.encode(start_text)]).to(device)

# torch.no_grad() — no gradients needed during inference, saves memory
with torch.no_grad():
    output_ids = generate(model, start_ids, max_new_tokens=20)

print(tokenizer.decode(output_ids[0].tolist()))

Every effort moves you forward. Every effort moves you forward. Every effort moves you forward. Every effort moves you
